### LOGIKAI KAPU

Az alábbi scriptek a Petőfi Irodalmi Múzeum OPAC felületéről letöltött rekordok összefésülését végzi.
1. A script a JSON formátumra alakított, leltári számmal kiegészített XML fájlt beolvassa, és struktuált XLSX formátumban visszadja.
2. Az XLSX fájlt a további műveletek elvégzésére visszaalakítja JSON formátumba.
3. A script ellenőrzi, hogy a lekérdezett kapcsolatokban vannak-e olyan új entitások, akik a nodes (namespace) fájlban nem szerepelnek. Ha egyezést talál, a hozzá tartozó kapcsolatokat kiszűri, ezek ugyanis egy korábbi lekérdezésből már bekerültek az edges fájlba.
4. Az új entitásokat tartalmazó kapcsolatokat kimenti egy new_entites.json fájlba.
5. A kimeneti JSON fájl XLSX formátumra konvertálását követően az új kapcsolatok hozzáilleszthetők a meglévőkhöz.

In [ ]:
import pandas as pd
import json

In [ ]:
koztes_json = "opac_records.json"
kimeneti_excel = "opac_records.xlsx"

# Beolvasás JSON-ból
with open(koztes_json, 'r', encoding='utf-8') as f:
    mentett_adatok = json.load(f)

# DataFrame építése
df = pd.DataFrame(mentett_adatok)

# Oszlopok sorrendbe rendezése
oszlop_sorrend = [
    'bib.rekord', 'leltari_szam', 'weight', 'date', 
    'megjegyzes', 'source', 'source_label', 'target', 'target_label'
]
df = df.reindex(columns=oszlop_sorrend)

# Mentés Excelbe
df.to_excel(kimeneti_excel, index=False)

print(f"Az Excel táblázat sikeresen legenerálva: {kimeneti_excel}")
df.head()

In [ ]:
df = pd.read_excel("namespace.xlsx", engine="openpyxl")
df = df.where(pd.notnull(df), None)
records = df.to_dict(orient="records")

with open("namespace.json", "w", encoding="utf-8") as f:
    json.dump(records, f, ensure_ascii=False, indent=2)

In [ ]:
# --- FÁJLNEVEK BEÁLLÍTÁSA ---
NAMESPACE_FILE = 'namespace.json'
RECORDS_FILE = 'kaffka_pim_opac.json'
OUTPUT_FILE = 'new_entities.json'

# Fájlok betöltése
with open(NAMESPACE_FILE, 'r', encoding='utf-8') as f:
    namespace = json.load(f)

with open(RECORDS_FILE, 'r', encoding='utf-8') as f:
    records = json.load(f)

print(f"Sikeresen betöltve: {len(namespace)} névtér elem és {len(records)} kapcsolat rekord.")

def extract_pure_name(label_str):
    """Levágja a zárójeles részt a név végéről (pl. 'Váry Rezső (1867-1940)' -> 'Váry Rezső')"""
    if not label_str:
        return ""
    if "(" in label_str:
        return label_str.split("(")[0].strip()
    return label_str.strip()

# 1. Összegyűjtjük a lokális névtérben létező összes ID-t (stringként az összehasonlíthatóság érdekében)
existing_ids = set()
for item in namespace:
    if isinstance(item, dict) and "id" in item:
        if item["id"] is not None:
            existing_ids.add(str(item["id"]).strip())

# 2. Végigmegyünk a kapcsolatokon és kigyűjtjük a hiányzó entitásokat
# Szótárként kell gyűjteni (ID alapján), hogy a duplikációkat automatikusan kiszűrjük
new_entities_dict = {}

for r in records:
    for role in ["source", "target"]:
        id_val = r.get(role)
        label_val = r.get(f"{role}_label")
        
        # Csak akkor nézzük, ha van azonosító és az nem üres vagy "MISSING"
        if id_val is not None and id_val != "MISSING":
            str_id = str(id_val).strip()
            
            # Ha az ID nincs benne a névtérben, és még nem adtuk hozzá a set-hez sem
            if str_id not in existing_ids and str_id not in new_entities_dict:
                pure_name = extract_pure_name(label_val)
                
                # Elkészítjük az új rekordot a kért struktúrával
                new_entities_dict[str_id] = {
                    "Halmaz": None,
                    "id": id_val,  # Megtartja az eredeti típust (szám vagy string)
                    "típus": None,
                    "label": pure_name,
                    "weight": None,
                    "log_suly": None,
                    "névvariáns": None,
                    "dátum": None,
                    "foglalkozás": None,
                    "megjegyzés a foglalkozásról": None
                }

# 3. Lista formátumra alakítás és mentés
new_entities_list = list(new_entities_dict.values())

with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
    json.dump(new_entities_list, f, ensure_ascii=False, indent=4)

print("-" * 50)
print(f"Talált új (hiányzó) entitások száma: {len(new_entities_list)}")
print(f"A kimenet elmentve ide: {OUTPUT_FILE}")
print("-" * 50)

In [ ]:
# Opcionális - A new_entities.json átalakítása XLSX formátumra

INPUT_FILE = "new_entities.json"
OUTPUT_FILE = "new_entities.xlsx"

with open(INPUT_FILE, "r", encoding="utf-8") as f:
    new_entities = json.load(f)

df_new_entities = pd.DataFrame(new_entities)
df_new_entities.to_excel(OUTPUT_FILE, index=False, engine="openpyxl")

print(f"Az XLSX fájl elkészült: {OUTPUT_FILE}")
df_new_entities.head()